In [1]:
import os, sys
from pathlib import Path 
sys.path.append(str(Path().resolve().parent))
from src.dataset.clustering_dataset import ClusteringDataset
import torch

In [ ]:
# 例: Notebook / スクリプト側
import sys
from pathlib import Path 
sys.path.append(str(Path().resolve().parent))
from src.dataset.clustering_dataset import ClusteringDataset, UnicodeClassMapper
from torch.utils.data import DataLoader

root_dir = r"C:\Users\kotat\MyPrograms\MyKuzushiji\kuzushiji-recognition\char_sep_datas"
root_dir = r"/scratch/users/grad/2025/25t0024/programs/MyKuzushiji/kuzushiji-recognition/char_sep_datas"

test_docs = [
    "200021637","100249371","100249537","200005598",
    "200014740","200020019","200021712","200021869",
]

# 1) まず train/test を作る（mapper を共有するとクラス次元Cが揃います）
mapper = UnicodeClassMapper()

train_ds = ClusteringDataset(
    root_dir=root_dir,
    canvas_width=2048,
    patch_size=256,
    test_mode=False,
    test_docs=test_docs,
    mapper=mapper,
)

test_ds = ClusteringDataset(
    root_dir=root_dir,
    canvas_width=2048,
    patch_size=256,
    test_mode=True,
    test_docs=test_docs,
    mapper=mapper,  # ★同じmapper
)

print("train:", len(train_ds), "test:", len(test_ds))
print("num_classes(C):", mapper.num_classes)

# 2) 1サンプル取り出し
sample = train_ds[6]
img = sample["image"]                 # (3,H,W) float32 0..1
final_mask = sample["final_text_mask"]# (H,W) 0/1
aff_mask = sample["affinity_mask"]    # (H,W) 0/1
label_map = sample["label_map"]       # (H,W,C) one-hot
meta = sample["meta"]

print(img.shape, final_mask.shape, aff_mask.shape, label_map.shape, meta)

# 3) DataLoader で回す
loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
batch = next(iter(loader))
print(batch["image"].shape)          # (B,3,H,W)
print(batch["label_map"].shape)      # (B,H ,W,C)

train: 5204 test: 947
num_classes(C): 159
torch.Size([3, 3328, 2048]) torch.Size([3328, 2048]) torch.Size([3328, 2048]) torch.Size([3328, 2048, 159]) {'doc_id': '100241706', 'image_id': '100241706_00004_2', 'image_path': 'C:\\Users\\kotat\\MyPrograms\\MyKuzushiji\\kuzushiji-recognition\\char_sep_datas\\100241706\\images\\100241706_00004_2.jpg', 'num_classes': 159}
torch.Size([1, 3, 3328, 2048])
torch.Size([1, 3328, 2048, 159])


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchTransformerSep(nn.Module):
    def __init__(self, img_channels=3, big_patch_size=256, vit_patch_size=16,
                 d_model=256, nhead=8, num_layers=4, num_classes=2):
        super().__init__()
        self.big_patch_size = big_patch_size     # 256
        self.vit_patch_size = vit_patch_size     # 16
        self.img_channels = img_channels

        inner_patch_dim = img_channels * vit_patch_size * vit_patch_size
        self.patch_embed = nn.Linear(inner_patch_dim, d_model)

        max_tokens = (big_patch_size // vit_patch_size) ** 2  # 256
        self.pos_embed = nn.Parameter(torch.zeros(1, max_tokens, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # ★ここが「出力層の形」: Cクラスぶん出す
        self.head = nn.Conv2d(d_model, num_classes, kernel_size=1)

    def forward(self, img):
        """
        img: (B, 3, H, W), H,W は 256 の倍数を想定
        return: logits: (B, C, H, W)
        """
        B, C, H, W = img.shape
        P = self.big_patch_size      # 256
        p = self.vit_patch_size      # 16

        assert H % P == 0 and W % P == 0, "H,W は big_patch_size(256) の倍数を想定"
        nH_big = H // P
        nW_big = W // P
        n_big = nH_big * nW_big

        # 1) 256×256 タイル化
        x = img.unfold(2, P, P).unfold(3, P, P)                 # (B,C,nH,nW,P,P)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()            # (B,nH,nW,C,P,P)
        x = x.view(B * n_big, C, P, P)                          # (B*n_big,C,256,256)

        # 2) タイル内を 16×16 パッチ(=token)化
        n_h_inner = P // p                                       # 16
        n_w_inner = P // p                                       # 16
        N_inner = n_h_inner * n_w_inner                          # 256

        x = x.unfold(2, p, p).unfold(3, p, p)                    # (B*n_big,C,16,16,p,p)
        x = x.contiguous().view(B * n_big, C, N_inner, p * p)    # (B*n_big,C,256,256)
        x = x.permute(0, 2, 1, 3).contiguous().view(B * n_big, N_inner, -1)

        # 3) Transformer
        x = self.patch_embed(x)                                  # (B*n_big,256,d_model)
        x = x + self.pos_embed[:, :N_inner, :]
        x = self.encoder(x)

        # 4) 16×16 に戻して -> Cクラスlogits -> 256×256へ拡大
        x = x.view(B * n_big, n_h_inner, n_w_inner, -1).permute(0, 3, 1, 2).contiguous()
        logits_small = self.head(x)                               # (B*n_big,C,16,16)
        logits_tile = F.interpolate(logits_small, size=(P, P), mode="bilinear", align_corners=False)

        # 5) タイルを敷き詰めて (B,C,H,W)
        logits_tile = logits_tile.view(B, nH_big, nW_big, -1, P, P)
        logits = logits_tile.permute(0, 3, 1, 4, 2, 5).contiguous().view(B, -1, H, W)
        return logits


def masked_bce_onehot_loss(logits_bchw: torch.Tensor, target_bchw: torch.Tensor) -> torch.Tensor:
    """
    target は one-hot (B,C,H,W)。未ラベル画素は全0。
    未ラベル画素（target.sum(C)==0）は損失から除外する。
    """
    # (B,1,H,W)
    labeled = (target_bchw.sum(dim=1, keepdim=True) > 0).float()

    loss_map = F.binary_cross_entropy_with_logits(
        logits_bchw, target_bchw, reduction="none"
    )  # (B,C,H,W)

    loss_map = loss_map * labeled  # broadcast
    denom = (labeled.sum() * logits_bchw.size(1)).clamp_min(1.0)
    return loss_map.sum() / denom


def bce_onehot_with_background(
    logits_bchw: torch.Tensor,
    target_bchw: torch.Tensor,
    bg_weight: float = 0.05,   # 背景の重み（0<bg_weight<=1）。小さくするほど背景の影響を弱める
) -> torch.Tensor:
    """
    target: one-hot (B,C,H,W)
      - 文字(ラベルあり)画素: どれか1チャネルが1（想定）
      - 背景画素: 全チャネル0（重要）
    """
    # (B,1,H,W) 背景=1, 前景=0
    is_bg = (target_bchw.sum(dim=1, keepdim=True) == 0).float()
    is_fg = 1.0 - is_bg

    # (B,1,H,W) の画素重み（Cへbroadcastされる）
    w = is_fg + bg_weight * is_bg

    loss_map = F.binary_cross_entropy_with_logits(
        logits_bchw, target_bchw, reduction="none"
    )  # (B,C,H,W)

    loss_map = loss_map * w

    denom = (w.sum() * logits_bchw.size(1)).clamp_min(1.0)
    return loss_map.sum() / denom


In [4]:
import os
from matplotlib import pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

C = train_ds.mapper.num_classes  # ★ClusteringDatasetのクラス次元
model = PatchTransformerSep(
    img_channels=3,
    big_patch_size=256,
    vit_patch_size=16,
    d_model=256,
    nhead=8,
    num_layers=4,
    num_classes=C,               # ★出力をCチャネルに
).to(device)
ckpt_dir = 'checkpoints_PatchTransformerSep_Clustering'
best_ckpt_path = os.path.join(ckpt_dir, "epoch_37.pth")
if not os.path.exists(best_ckpt_path):
    raise FileNotFoundError(f"best checkpoint not found: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
with torch.no_grad():
    for step, batch in enumerate(test_loader, start=1):
        
        img = batch["image"].to(device)
        label_map = batch["label_map"].to(device)
        target = label_map.permute(0, 3, 1, 2).float().contiguous()

        logits = model(img)                 # ★これが必要
        prob = torch.sigmoid(logits) 

        # GT（one-hot）: クラス方向に和（基本 0/1 のマスク）
        # ...existing code...
        target_sum = target.sum(dim=1)  # (B,H,W)

        print("target_sum min/max:", float(target_sum.min()), float(target_sum.max()))
        print("background ratio (sum==0):", float((target_sum == 0).float().mean()))
        # ...existing code...

        # 「合計」を見たいなら sum（0..C）
        prob_sum = prob.sum(dim=1)      # (B,H,W)

        # 「どれかのクラスが高い」を見たいなら max（0..1）
        prob_max = prob.max(dim=1).values  # (B,H,W)

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(img[0].permute(1,2,0).detach().cpu().numpy())
        plt.title("Input Image")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(target_sum[0].detach().cpu().numpy(), cmap="jet", vmin=0, vmax=1)
        plt.title("GT: sum over classes")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(prob_max[0].detach().cpu().numpy(), cmap="jet", vmin=0, vmax=1)
        plt.title("Pred: max prob over classes (sigmoid)")
        plt.axis("off")

        plt.tight_layout()
        plt.show()
        # print(f"[test] step {step} loss {loss.item():.6f}")

c:\Users\kotat\AppData\Local\anaconda3\envs\kuzushiji\Lib\site-packages\torch\nn\modules\transformer.py:685: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  return torch._transformer_encoder_layer_fwd(


target_sum min/max: 0.0 0.0
background ratio (sum==0): 1.0


KeyboardInterrupt: 

In [5]:
# ==== Evaluation cell: mIoU (argmax, ignore unlabeled pixels) ====
import math
import numpy as np
import torch
import torch.nn.functional as F

def _safe_div(num: torch.Tensor, den: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    return num / den.clamp_min(eps)

def _align_logits_target(logits_bchw: torch.Tensor, target_bchw: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Align shapes between logits and target.
    - If spatial size differs, resize target to logits size (nearest).
    - If channel size differs, use common min(C).
    """
    _, c_log, h_log, w_log = logits_bchw.shape
    _, c_tgt, h_tgt, w_tgt = target_bchw.shape

    if (h_log != h_tgt) or (w_log != w_tgt):
        target_bchw = F.interpolate(target_bchw, size=(h_log, w_log), mode="nearest")

    if c_log != c_tgt:
        c = min(c_log, c_tgt)
        logits_bchw = logits_bchw[:, :c]
        target_bchw = target_bchw[:, :c]
    return logits_bchw, target_bchw

@torch.no_grad()
def miou_argmax_labeled(logits_bchw: torch.Tensor, target_bchw: torch.Tensor) -> float:
    """
    logits_bchw : (B,C,H,W) 生logit（sigmoid/softmax前でもOK。argmaxは単調変換で不変）
    target_bchw : (B,C,H,W) one-hot想定。背景/未ラベルは全0。
    mIoUは「labeled画素（target.sum>0）」のみで計算する。
    """
    logits_bchw, target_bchw = _align_logits_target(logits_bchw, target_bchw)
    B, C, H, W = logits_bchw.shape

    labeled = (target_bchw.sum(dim=1) > 0)  # (B,H,W) bool

    pred_idx = logits_bchw.argmax(dim=1)     # (B,H,W)
    gt_idx   = target_bchw.argmax(dim=1)     # (B,H,W) ※ unlabeledは0になるので labeled で無視

    ious = []
    for c in range(C):
        gt_c = (gt_idx == c) & labeled
        if not bool(gt_c.any()):
            continue  # GTに出ないクラスはmacroから除外
        pr_c = (pred_idx == c) & labeled
        inter = (gt_c & pr_c).sum().float()
        union = (gt_c | pr_c).sum().float()
        ious.append(float(_safe_div(inter, union).item()))

    return float(np.mean(ious)) if len(ious) else float("nan")

# ===== run over test_loader =====
miou_list = []

model.eval()
with torch.no_grad():
    for step, batch in enumerate(test_loader, start=1):
        img = batch["image"].to(device)
        label_map = batch["label_map"].to(device)
        target = label_map.permute(0, 3, 1, 2).float().contiguous()  # (B,C,H,W)

        logits = model(img)  # (B,C,H,W)
        miou = miou_argmax_labeled(logits, target)
        miou_list.append(miou)

        if step <= 3:
            print(f"[step {step}] mIoU={miou:.4f}")

def _nanmean(xs):
    xs = [x for x in xs if not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(xs)) if len(xs) else float("nan")

print("==== TEST SUMMARY (mIoU) ====")
print("mIoU (argmax, labeled):", _nanmean(miou_list))

[step 1] mIoU=nan
[step 2] mIoU=0.9927
[step 3] mIoU=0.7480


KeyboardInterrupt: 